# 01 · 注意力机制：Q/K/V 与缩放点积注意力  Attention: Q/K/V & Scaled Dot-Product

> **能力标签**：GA1（Transformer 架构 / Transformer Architecture）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**查询**（Query, Q）、**键**（Key, K）、**值**（Value, V）的直觉含义与矩阵维度。
2. 推导并手写**缩放点积注意力**（Scaled Dot-Product Attention）公式：$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\dfrac{QK^T}{\sqrt{d_k}}\right)V$
3. 解释缩放因子 $\frac{1}{\sqrt{d_k}}$ 的必要性。
4. 实现**因果掩码**（causal mask）和 **padding 掩码**（padding mask），并理解在 softmax 前置 $-\infty$ 的原因。
5. 可视化注意力热图（attention heatmap），识别不同掩码模式。

> 对应能力：**GA1**

## 概念讲解 Concepts

### 1. Q / K / V 的直觉 Intuition

Attention 机制可以类比为一个**软检索**（soft retrieval）过程：

| 符号 | 全称 | 直觉 |
|------|------|------|
| Q (Query) | 查询向量 | "我想要什么" |
| K (Key)   | 键向量   | "每个位置能提供什么" |
| V (Value) | 值向量   | "每个位置实际携带的信息" |

给定序列长度 $T$、模型维度 $d$：$Q, K, V \in \mathbb{R}^{T \times d}$。

---

### 2. 缩放点积注意力 Scaled Dot-Product Attention

$$\text{scores} = \frac{QK^T}{\sqrt{d_k}} \in \mathbb{R}^{T \times T}$$

$$\text{attn} = \text{softmax}(\text{scores}) \in \mathbb{R}^{T \times T}$$

$$\text{out} = \text{attn} \cdot V \in \mathbb{R}^{T \times d}$$

**为什么要除以 $\sqrt{d_k}$？**  
当 $d_k$ 较大时，点积的方差随 $d_k$ 线性增长。除以 $\sqrt{d_k}$ 将其归一化，防止 softmax 进入梯度几乎为零的饱和区。

---

### 3. 掩码 Masks

两种常用掩码，在 softmax **之前**将不合法位置的得分置为 $-\infty$：

**因果掩码（Causal Mask / 上三角掩码）**  
用于**自回归生成**（语言模型），防止当前 token 看到未来 token：

```
位置 0 可见: [0]
位置 1 可见: [0, 1]
位置 2 可见: [0, 1, 2]
```

**Padding 掩码（Padding Mask）**  
用于**批处理变长序列**，将填充（PAD）位置屏蔽，令其注意力权重为 0。

---

### 4. Softmax 的数值稳定性 Numerical Stability

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

将被遮掩位置置为 $-\infty$ 后，$e^{-\infty} = 0$，对应注意力权重精确为 0。

---

### 5. 注意力热图 Attention Heatmap

注意力矩阵 $\text{attn} \in [0,1]^{T \times T}$ 可直接可视化：颜色越深，该位置对输出的贡献越大。不同掩码模式呈现出截然不同的几何形状（上三角 vs. 矩形块）。

## 示例 Worked Example

In [ ]:
# ── 缩放点积注意力（镜像 w7-attention）─────────────────────────────────────
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)


def scaled_dot_product_attention(q, k, v, mask=None):
    """缩放点积注意力。q, k, v: (..., T, d)。返回 (输出, 注意力权重)。
    mask: 可选 (T, T) 布尔/0-1，0 处不允许注意（置 -inf）。
    Scaled dot-product attention; mask==0 positions get -inf before softmax.
    """
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)   # (..., T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = F.softmax(scores, dim=-1)                   # (..., T, T)
    return attn @ v, attn                              # (output, weights)


# ── 基础前向传播演示 Basic forward pass ──────────────────────────────────────
T, d = 5, 8   # 序列长度, 注意力维度
q = torch.randn(T, d)
k = torch.randn(T, d)
v = torch.randn(T, d)

out, attn_weights = scaled_dot_product_attention(q, k, v)
print("无掩码注意力 (No mask):")
print(f"  q.shape={q.shape}, k.shape={k.shape}, v.shape={v.shape}")
print(f"  out.shape={out.shape}       (应为 ({T}, {d}))")
print(f"  attn_weights.shape={attn_weights.shape}  (应为 ({T}, {T}))")
print(f"  attn_weights 每行之和={attn_weights.sum(dim=-1).tolist()}  (应全为 1.0)")
assert out.shape == (T, d)
assert attn_weights.shape == (T, T)
assert torch.allclose(attn_weights.sum(dim=-1), torch.ones(T), atol=1e-5)
print("✓ 形状与归一化正确 Shape and normalization OK")


In [ ]:
# ── 因果掩码演示 Causal Mask Demo ────────────────────────────────────────────
# 上三角掩码：位置 i 只能注意 j <= i 的 token

def causal_mask(T):
    """生成 T×T 因果掩码（下三角为 1，上三角为 0）。
    Lower-triangular mask: position i may attend to j <= i only.
    """
    return torch.tril(torch.ones(T, T, dtype=torch.bool))

mask_c = causal_mask(T)
print("因果掩码 Causal mask (T=5):")
print(mask_c.int())

out_c, attn_c = scaled_dot_product_attention(q, k, v, mask=mask_c)
print(f"\n因果注意力权重矩阵 (上三角应为 0):")
print(attn_c.detach().round(decimals=3))

# 验证上三角（未来位置）权重为 0
upper = attn_c[torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)]
assert torch.allclose(upper, torch.zeros_like(upper), atol=1e-6), "因果掩码失败 Causal mask failed"
print("✓ 因果掩码正确：所有未来位置权重为 0 Causal mask OK: future weights are 0")


In [ ]:
# ── Padding 掩码演示 Padding Mask Demo ─────────────────────────────────────
# 批处理中序列长度不同，用 PAD token 填充。掩码遮蔽 PAD 位置。

def padding_mask(seq_len, max_len):
    """生成 1D padding mask，前 seq_len 个位置为 True，其余为 False。
    Returns a max_len×max_len mask where both row AND col i,j are valid
    only if both i<seq_len and j<seq_len.
    """
    # 有效列掩码：(max_len,) → (max_len, max_len)
    valid_col = torch.arange(max_len) < seq_len   # (max_len,)
    return valid_col.unsqueeze(0).expand(max_len, max_len)  # (max_len, max_len)


T_max = 5
# 模拟序列实际长度为 3（位置 3,4 是 PAD）
real_len = 3
mask_p = padding_mask(real_len, T_max)
print("Padding 掩码 (实际长度=3, 最大=5):")
print(mask_p.int())

out_p, attn_p = scaled_dot_product_attention(q, k, v, mask=mask_p)
print(f"\nPadding 注意力权重（第 {real_len} 列及之后应为 0）:")
print(attn_p.detach().round(decimals=3))

# PAD 列（j >= real_len）应全为 0
assert torch.allclose(attn_p[:, real_len:], torch.zeros(T_max, T_max - real_len), atol=1e-6)
print("✓ Padding 掩码正确：PAD 列权重为 0 Padding mask OK: PAD columns are 0")


In [ ]:
# ── 批量输入（batch + heads）演示 Batched Input Demo ──────────────────────
# 真实 MHA 中 q/k/v 形状为 (B, H, T, d_head)

B, H = 2, 4
d_head = 8
T = 5

torch.manual_seed(7)
q_b = torch.randn(B, H, T, d_head)
k_b = torch.randn(B, H, T, d_head)
v_b = torch.randn(B, H, T, d_head)

mask_c_b = causal_mask(T)   # 广播到 (B, H, T, T)
out_b, attn_b = scaled_dot_product_attention(q_b, k_b, v_b, mask=mask_c_b)

print("批量多头输入 Batched multi-head:")
print(f"  q_b.shape={q_b.shape}  →  out_b.shape={out_b.shape}")
print(f"  attn_b.shape={attn_b.shape}  (应为 ({B}, {H}, {T}, {T}))")
assert out_b.shape == (B, H, T, d_head)
assert attn_b.shape == (B, H, T, T)
print("✓ 批量注意力形状正确 Batched attention shapes OK")


In [ ]:
# ── 注意力热图可视化 Attention Heatmap (matplotlib Agg) ───────────────────
import os
import tempfile
import matplotlib
matplotlib.use("Agg")   # 无显示器后端 Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

torch.manual_seed(99)
T_vis = 6
labels = [f"t{i}" for i in range(T_vis)]

q_v = torch.randn(T_vis, 16)
k_v = torch.randn(T_vis, 16)
v_v = torch.randn(T_vis, 16)

# 无掩码 + 因果掩码
_, attn_none = scaled_dot_product_attention(q_v, k_v, v_v)
_, attn_causal = scaled_dot_product_attention(q_v, k_v, v_v, mask=causal_mask(T_vis))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, attn, title in zip(
        axes,
        [attn_none.detach(), attn_causal.detach()],
        ["无掩码注意力 (No Mask)", "因果掩码注意力 (Causal Mask)"]):
    im = ax.imshow(attn.numpy(), vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_xticks(range(T_vis))
    ax.set_yticks(range(T_vis))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Key (键位置)", fontsize=9)
    ax.set_ylabel("Query (查询位置)", fontsize=9)
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()

# 保存到临时目录（不调用 plt.show()）Save to tempdir, no plt.show()
tmpdir = tempfile.mkdtemp()
fig_path = os.path.join(tmpdir, "attention_heatmap.png")
fig.savefig(fig_path, dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"注意力热图已保存 Attention heatmap saved: {fig_path}")
print("  左图（无掩码）：全矩阵均有权重；右图（因果掩码）：上三角为零（蓝色色块仅出现在下三角）")


## 动手 Exercise

In [ ]:
# ── 动手练习：实现带温度参数的注意力 Temperature-Scaled Attention ───────────
# 任务：修改注意力计算，引入温度参数 temperature，使得
#       scores = Q @ K.T / (sqrt(d_k) * temperature)
# temperature > 1 → 注意力更"平滑"（分散）
# temperature < 1 → 注意力更"尖锐"（集中）
# 验证：temperature=0.1 时，注意力权重的熵（entropy）应低于 temperature=10 时。

import math

torch.manual_seed(2024)
T_ex, d_ex = 4, 16
q_ex = torch.randn(T_ex, d_ex)
k_ex = torch.randn(T_ex, d_ex)
v_ex = torch.randn(T_ex, d_ex)


def temp_attention(q, k, v, temperature=1.0):
    """带温度参数的缩放点积注意力。
    Scaled dot-product attention with temperature scaling.
    temperature > 1: softer (more uniform); temperature < 1: sharper.
    """
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / (math.sqrt(d) * temperature)
    attn = F.softmax(scores, dim=-1)
    return attn @ v, attn


def entropy(attn):
    """计算注意力矩阵的平均熵（每行）。Compute mean row-entropy of attention matrix."""
    # 避免 log(0)
    safe = attn.clamp(min=1e-9)
    return -(safe * safe.log()).sum(dim=-1).mean().item()


_, attn_sharp = temp_attention(q_ex, k_ex, v_ex, temperature=0.1)
_, attn_soft  = temp_attention(q_ex, k_ex, v_ex, temperature=10.0)

ent_sharp = entropy(attn_sharp)
ent_soft  = entropy(attn_soft)

print(f"temperature=0.1  → 平均熵 entropy = {ent_sharp:.4f}  (尖锐 sharp)")
print(f"temperature=10.0 → 平均熵 entropy = {ent_soft:.4f}  (平滑 soft)")
assert ent_sharp < ent_soft, "temperature 越小，熵应越低 (sharper → lower entropy)"
print("✓ 温度参数效果验证通过 Temperature scaling verified")


## 延伸阅读 Further Reading

1. **Vaswani et al. "Attention Is All You Need" (2017)**：<https://arxiv.org/abs/1706.03762> — 原始 Transformer 论文；第 3 节详述缩放点积注意力与多头注意力。
2. **The Annotated Transformer (Harvard NLP)**：<https://nlp.seas.harvard.edu/annotated-transformer/> — 带逐行注释的 PyTorch 实现，与本课示例直接对应。
3. **Karpathy "Let's build GPT" (YouTube)**：<https://www.youtube.com/watch?v=kCc8FmEb1nY> — 手写 GPT-2 及其因果注意力；演示因果掩码在自回归生成中的作用。
4. **Bahdanau et al. "Neural Machine Translation by Jointly Learning to Align and Translate" (2015)**：<https://arxiv.org/abs/1409.0473> — 注意力机制的早期来源，加性注意力（Additive Attention）。
5. **Illustrated Transformer (Jay Alammar)**：<https://jalammar.github.io/illustrated-transformer/> — 直观图解 Q/K/V 与多头注意力，适合初学者。

## 关联练习 Related Assignments

```bash
python check.py w7-attention
```

> 实现 `scaled_dot_product_attention(q, k, v, mask=None)`，支持因果掩码与 padding 掩码。

> 能力标签：**GA1** · threshold ≥ 0.7